In [38]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/raw/AirBnB_BandungRaya_All.csv')

# Pembersihan harga & rating 
df['harga'] = df['harga'].str.replace('Rp', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(int)
df['rating'] = pd.to_numeric(df['rating'].replace('New', np.nan), errors='coerce')

df.head(2)

,nama,harga,rating,jumlah_review,link,wilayah
0,Home in Cibeunying Kaler - Serene Pet-Friendly...,9848899,5.00,33,https://www.airbnb.com/rooms/1647712818622104523,Kota Bandung
1,"Apartment in Cidadap - Dago, Ciumbuleuit, ITB ...",3446781,4.98,132,https://www.airbnb.com/rooms/1583745571091269827,Kota Bandung


In [39]:
def extract_tipe(nama):
    # Mengambil teks sebelum kata ' in '
    match = re.search(r'^(.*?)\s+in\s+', str(nama), re.IGNORECASE)
    return match.group(1).strip() if match else 'Other'

def extract_lokasi(nama):
    # Mengambil teks di antara ' in ' dan ' - '
    match = re.search(r'\s+in\s+(.*?)\s+-', str(nama), re.IGNORECASE)
    return match.group(1).strip() if match else 'Unknown'

df['tipe_raw'] = df['nama'].apply(extract_tipe)
df['lokasi_raw'] = df['nama'].apply(extract_lokasi)

print("Jumlah Tipe Properti:")
print(df['tipe_raw'].value_counts().head(10))

print("\nJumlah Lokasi (Kecamatan/Area) Terbanyak:")
print(df['lokasi_raw'].value_counts().head(10))

Jumlah Tipe Properti:
tipe_raw
Home           250
Villa          127
Apartment      101
Room            19
Guesthouse      11
Condo           10
Cabin           10
Rooms            9
Tiny home        7
Guest suite      5
Name: count, dtype: int64

Jumlah Lokasi (Kecamatan/Area) Terbanyak:
lokasi_raw
Megamendung          68
Cisarua              51
Parompong            35
Lembang              24
Cimenyan             23
Cicendo              21
Bogor Selatan        21
Cidadap              16
Kecamatan Cidadap    16
Babakan Madang       15
Name: count, dtype: int64


In [30]:
test1 = df.to_csv('../data/processed/data_cleaned.csv', index=False)

In [41]:
# Ambil semua lokasi unik
lokasi_unik = df['lokasi_raw'].value_counts().reset_index()
lokasi_unik.columns = ['lokasi_raw', 'jumlah']

print(f"Total lokasi unik: {len(lokasi_unik)}")
lokasi_unik.to_csv('../data/reference/daftar_lokasi_mentah.csv', index=False)

Total lokasi unik: 82


In [42]:
print(df.columns.tolist())
mapping = pd.read_csv('../data/reference/kecamatan_mapping.csv')
print(mapping.columns.tolist())

['nama', 'harga', 'rating', 'jumlah_review', 'link', 'wilayah', 'tipe_raw', 'lokasi_raw']
['lokasi_raw', '\tkecamatan', '\tkabupaten_kota', '\tregion']


In [ ]:
# Load mapping
mapping = pd.read_csv('../data/reference/kecamatan_mapping.csv')

# Validasi mapping
missing_lokasi = set(df['lokasi_raw']) - set(mapping['lokasi_raw'])
if missing_lokasi:
    print(f"Ada sebanyak {len(missing_lokasi)} lokasi yang tidak ada di mapping:")
    for lokasi in missing_lokasi:
        print(f"- {lokasi}")

# Cek missing
missing_lokasi = set(df['lokasi_raw']) - set(mapping['lokasi_raw'])
if missing_lokasi:
    print(f"Ada sebanyak {len(missing_lokasi)} lokasi yang tidak ada di mapping:")
    for lokasi in missing_lokasi:
        print(f"- {lokasi}")

# Merge mapping ke df
df = df.merge(mapping, on='lokasi_raw', how='left')
print(df.head())

In [48]:
# Cek apakah ada gagal merge
missing_region = df[df['region'].isna()]
print(f"Jumlah baris tanpa region: {len(missing_region)}")

# Bersihkan karakter tab (\t) di kolom kecamatan, kabupaten, dan region
df.columns = df.columns.str.replace('\t','').str.strip()  

# Cek kolom setelah merge
print(df.columns.tolist())

Jumlah baris tanpa region: 0
['nama', 'harga', 'rating', 'jumlah_review', 'link', 'wilayah', 'tipe_raw', 'lokasi_raw', 'kecamatan', 'kabupaten_kota', 'region']


## Ringkasan Progress Cleaning (WIP)

- **Ekstraksi Tipe & Lokasi**: Berhasil mengekstrak `tipe_raw` (Home, Villa, Apartment, dll) dan `lokasi_raw` dari kolom `nama` menggunakan regex.
- **Mapping Kecamatan**: Berhasil menggabungkan data dengan `kecamatan_mapping.csv` untuk mendapatkan kolom `kecamatan` dan `region` (Bandung Raya vs Bogor). Karakter `\t` pada nama kolom sudah dibersihkan.
- **Keputusan Tertunda (Next Steps)**: 
  - Mengevaluasi hasil *sanity check* (crosstab) antara `tipe_raw` dan fasilitas. Jika persentasenya tidak masuk akal (bias), fitur fasilitas akan dibuang dan hanya menggunakan `bedrooms` serta `tipe_raw`.
  - Melakukan transformasi logaritma (`log_harga`) dan perhitungan Bayesian rating (`rating_adj`).
  - Menyimpan dataset final ke `data/processed/airbnb_clean.csv`.